# Plackett–Burman screening and ANOVA reconstruction

This notebook reproduces the main calculations shown in the presentation for the paper **"An Engineering Framework for Adaptive Winglet Design"**.

It computes:

- contrasts
- effects
- sums of squares
- ANOVA quantities
- a worked example for **height** \(X_4\)
- the right-tail **F-distribution** area for the selected factor

## Inputs

You can modify the `CD` values below and rerun the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Try scipy first. If unavailable in a given JupyterLite build, the notebook still runs,
# but p-values and the F-density plot will need scipy.
try:
    from scipy.stats import f as f_dist
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

In [ ]:
# Coded 12-run PB design used in the slides
factors = ["X1_root_chord", "X2_sweep", "X3_taper", "X4_height", "X5_cant", "X6_toe", "X7_twist"]

coded = np.array([
    [ 1,  1,  1,  1, -1,  1, -1],
    [ 1, -1, -1,  1,  1, -1, -1],
    [ 1, -1, -1, -1,  1,  1,  1],
    [-1, -1,  1, -1,  1,  1, -1],
    [-1, -1,  1, -1, -1, -1,  1],
    [-1,  1, -1, -1, -1,  1, -1],
    [ 1,  1, -1,  1, -1, -1,  1],
    [ 1,  1,  1,  1,  1,  1,  1],
    [ 1, -1,  1, -1, -1, -1, -1],
    [-1,  1, -1,  1,  1,  1, -1],
    [-1,  1,  1, -1,  1, -1,  1],
    [-1, -1, -1,  1, -1, -1,  1],
], dtype=int)

# Drag coefficient values from the slides / paper
CD = np.array([
    0.020126,
    0.023634,
    0.023588,
    0.022778,
    0.022989,
    0.023436,
    0.023351,
    0.023626,
    0.022702,
    0.020402,
    0.023475,
    0.021289
], dtype=float)

df = pd.DataFrame(coded, columns=factors)
df.insert(0, "run", np.arange(1, 13))
df["CD"] = CD
df

## Core formulas

For each factor \(X_i\):

\[
\mathrm{Contrast}_{X_i}=\sum s_{r,i} y_r
\]

\[
\mathrm{Effect}(X_i)=\frac{\mathrm{Contrast}_{X_i}}{6}
\]

\[
SS_{X_i}=\frac{\mathrm{Contrast}_{X_i}^2}{12}
\]

For the full PB main-effects model:

\[
SS_T = \sum_{r=1}^{12}(y_r-\bar y)^2
\]

\[
SS_M = \sum_{i=1}^{7} SS_{X_i}
\]

\[
SS_E = SS_T - SS_M
\]

\[
MS_E = \frac{SS_E}{df_E}, \qquad df_E = 4
\]

\[
F_{X_i}=\frac{MS_{X_i}}{MS_E}
\]

\[
p_{X_i}=P(F_{1,4}\ge F_{X_i}\mid H_0)
\]

In [ ]:
N = len(CD)
mean_cd = CD.mean()

results = []
for j, name in enumerate(factors):
    signs = coded[:, j]
    contrast = float(np.sum(signs * CD))
    effect = contrast / 6.0
    ss = contrast**2 / 12.0
    results.append({
        "factor": name,
        "contrast": contrast,
        "effect": effect,
        "SS": ss,
        "df": 1
    })

results_df = pd.DataFrame(results)

SS_total = float(np.sum((CD - mean_cd)**2))
SS_model = float(results_df["SS"].sum())
df_total = N - 1
df_model = len(factors)
df_error = df_total - df_model
SS_error = SS_total - SS_model
MS_error = SS_error / df_error

results_df["MS"] = results_df["SS"] / results_df["df"]
results_df["F"] = results_df["MS"] / MS_error

if SCIPY_AVAILABLE:
    results_df["p_value"] = results_df["F"].apply(lambda x: float(f_dist.sf(x, 1, df_error)))
else:
    results_df["p_value"] = np.nan

summary = pd.DataFrame({
    "quantity": ["N", "mean(CD)", "SS_total", "SS_model", "SS_error", "df_total", "df_model", "df_error", "MS_error"],
    "value": [N, mean_cd, SS_total, SS_model, SS_error, df_total, df_model, df_error, MS_error]
})

summary

In [ ]:
results_df

## Worked example: height \(X_4\)

In [ ]:
factor_name = "X4_height"
j = factors.index(factor_name)
signs = coded[:, j]

plus_runs = df.loc[signs == 1, ["run", "CD"]].copy()
minus_runs = df.loc[signs == -1, ["run", "CD"]].copy()

sum_plus = plus_runs["CD"].sum()
sum_minus = minus_runs["CD"].sum()
contrast_x4 = sum_plus - sum_minus
effect_x4 = contrast_x4 / 6
ss_x4 = contrast_x4**2 / 12
ms_x4 = ss_x4
f_x4 = ms_x4 / MS_error

if SCIPY_AVAILABLE:
    p_x4 = float(f_dist.sf(f_x4, 1, df_error))
else:
    p_x4 = None

print("High-level runs (X4 = +1)")
display(plus_runs)

print("Low-level runs (X4 = -1)")
display(minus_runs)

print(f"Sum at +1: {sum_plus:.6f}")
print(f"Sum at -1: {sum_minus:.6f}")
print(f"Contrast_X4 = {contrast_x4:.6f}")
print(f"Effect(X4)   = {effect_x4:.6f}")
print(f"SS_X4        = {ss_x4:.10f}")
print(f"MS_X4        = {ms_x4:.10f}")
print(f"MS_error     = {MS_error:.10f}")
print(f"F_X4         = {f_x4:.6f}")
if p_x4 is not None:
    print(f"p_X4         = {p_x4:.6f}")
else:
    print("p_X4         = requires scipy.stats.f.sf in this environment")

## Optional: choose a factor to inspect

In [ ]:
selected_factor = "X4_height"  # change this string and rerun
j = factors.index(selected_factor)

contrast = float(np.sum(coded[:, j] * CD))
effect = contrast / 6
ss = contrast**2 / 12
ms = ss
f_value = ms / MS_error

print(f"Selected factor: {selected_factor}")
print(f"Contrast = {contrast:.6f}")
print(f"Effect   = {effect:.6f}")
print(f"SS       = {ss:.10f}")
print(f"MS       = {ms:.10f}")
print(f"F        = {f_value:.6f}")

if SCIPY_AVAILABLE:
    p_value = float(f_dist.sf(f_value, 1, df_error))
    print(f"p-value  = {p_value:.6f}")
else:
    p_value = None
    print("p-value  = requires scipy.stats.f.sf in this environment")

## F-distribution plot

The shaded area to the right of the observed \(F\)-value is the **p-value**.

In [ ]:
x = np.linspace(0.001, 20, 600)

plt.figure(figsize=(8, 4.5))
if SCIPY_AVAILABLE:
    y = f_dist.pdf(x, 1, df_error)
    plt.plot(x, y, label="F(1, 4)")
    plt.axvline(f_value, linestyle="--", label=f"Observed F = {f_value:.3f}")
    mask = x >= f_value
    plt.fill_between(x[mask], y[mask], alpha=0.3, label="right-tail area = p-value")
else:
    plt.axvline(f_value, linestyle="--", label=f"Observed F = {f_value:.3f}")
    plt.text(0.5, 0.5, "scipy not available in this JupyterLite build", transform=plt.gca().transAxes, ha="center")

plt.xlabel("F")
plt.ylabel("Density")
plt.title("Right-tail probability of the F-distribution")
plt.legend()
plt.show()

## Reduced model after removing the least significant factor

The paper removes **toe angle** and repeats the regression.  
This notebook keeps the full PB-screening reconstruction as the main example.